# Function Approximation in Reinforcement Learning
## Tile Coding, RBF Features, and Semi-Gradient Methods

**What You'll Learn:**
- Why tabular methods fail in large/continuous state spaces
- Tile coding: overlapping grids for feature construction (from scratch)
- Radial Basis Function (RBF) features (from scratch)
- Semi-gradient TD(0) for value prediction
- Semi-gradient SARSA for control
- Solving MountainCar with linear function approximation

**Prerequisites:** TD learning (Notebook 3), basic optimization (gradient descent).

**References:**
- Sutton & Barto, *Reinforcement Learning: An Introduction*, Chapters 9-10
- Albus 1975, "A New Approach to Manipulator Control: The Cerebellar Model Articulation Controller (CMAC)"

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm
from mpl_toolkits.mplot3d import Axes3D
import gymnasium as gym
from itertools import product
from typing import List, Tuple, Optional
import warnings
warnings.filterwarnings('ignore')

# ── Reproducibility & plotting defaults ──
SEED = 42
np.random.seed(SEED)

plt.rcParams.update({
    'figure.figsize': (12, 5),
    'font.size': 12,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'lines.linewidth': 2,
})

# ── Color palette ──
COLORS = {
    'blue': 'steelblue',
    'red': 'coral',
    'green': 'seagreen',
    'yellow': 'goldenrod',
    'purple': 'mediumpurple',
}

# ── RL constants ──
GAMMA = 1.0
ALPHA = 0.01
EPSILON = 0.1
N_EPISODES = 1000

print('Setup complete.')

---
## 1. The Curse of Dimensionality

Tabular RL stores one value for every state (or state-action pair). A Q-table
has $|\mathcal{S}| \times |\mathcal{A}|$ entries. When the state space is
continuous or very large, storing and updating every entry becomes impossible.

**Function approximation** replaces the table with a parameterized function:

$$\hat{v}(s, \mathbf{w}) \approx v_\pi(s)$$

The simplest and most well-understood form is **linear approximation**:

$$\boxed{\hat{v}(s, \mathbf{w}) = \mathbf{w}^\top \mathbf{x}(s)}$$

where $\mathbf{x}(s) \in \mathbb{R}^d$ is a **feature vector** constructed from
the raw state $s$, and $\mathbf{w} \in \mathbb{R}^d$ is a learned weight vector.

The quality of learning depends critically on the choice of features
$\mathbf{x}(s)$. Two popular hand-crafted feature schemes are **tile coding**
and **radial basis functions (RBFs)**.

---
## 2. Semi-Gradient Methods

In standard supervised learning we minimize a loss by computing its full
gradient with respect to $\mathbf{w}$. In RL, the TD target
$R + \gamma \hat{v}(S', \mathbf{w})$ itself depends on $\mathbf{w}$, so the
true gradient of the squared TD error includes terms from differentiating the
target. **Semi-gradient methods** deliberately ignore the dependency of the
target on $\mathbf{w}$, treating it as a fixed constant.

### Semi-Gradient TD(0) Update

$$\boxed{\mathbf{w} \leftarrow \mathbf{w} + \alpha \left[ R + \gamma \hat{v}(S', \mathbf{w}) - \hat{v}(S, \mathbf{w}) \right] \nabla_{\mathbf{w}} \hat{v}(S, \mathbf{w})}$$

For the linear case $\hat{v}(s, \mathbf{w}) = \mathbf{w}^\top \mathbf{x}(s)$,
the gradient is simply:

$$\nabla_{\mathbf{w}} \hat{v}(S, \mathbf{w}) = \mathbf{x}(S)$$

So the weight update becomes:

$$\mathbf{w} \leftarrow \mathbf{w} + \alpha \, \delta \, \mathbf{x}(S)$$

where $\delta = R + \gamma \hat{v}(S', \mathbf{w}) - \hat{v}(S, \mathbf{w})$ is
the TD error.

---
## 3. Tile Coding

Tile coding (Albus, 1975) partitions the state space with **multiple
overlapping grids** (tilings). Each tiling divides the space into tiles; a state
activates exactly one tile per tiling.

Key properties:
- The feature vector $\mathbf{x}(s)$ is **binary** — each component is 0 or 1.
- The number of active features always equals the number of tilings.
- Different tilings are **offset** from each other, creating a form of coarse
  coding that enables smooth generalization.
- With $n$ tilings of $m$ tiles each (per dimension), the total number of
  features is $n \times m^{\text{dim}}$ (for a $\text{dim}$-dimensional space).

The offset for tiling $k$ is typically:

$$\text{offset}_k = \frac{k}{n} \times \text{tile\_width}$$

This ensures each tiling "sees" the space from a slightly different angle.

---
## 4. Radial Basis Function (RBF) Features

RBF features place Gaussian kernels at fixed centers $c_i$ in state space:

$$\boxed{x_i(s) = \exp\left(-\frac{\|s - c_i\|^2}{2\sigma^2}\right)}$$

Properties:
- Features are **continuous** values in $(0, 1]$.
- Provide **soft generalization**: nearby states share similar feature
  activations, smoothly falling off with distance.
- The width parameter $\sigma$ controls the generalization radius.
- We normalize the feature vector so that components sum to 1, giving a
  probability-like distribution over centers.

Compared to tile coding's hard boundaries, RBF features produce smoother value
functions but can be more expensive for high-dimensional spaces.

---
## 5. The MountainCar Environment

We use Gymnasium's `MountainCar-v0` as our testbed:

| Property | Value |
|----------|-------|
| State | (position $\in [-1.2, 0.6]$, velocity $\in [-0.07, 0.07]$) |
| Actions | 0 = push left, 1 = no push, 2 = push right |
| Goal | reach position $\geq 0.5$ |
| Max steps | 200 |
| Reward | $-1$ per time step |

The car is too weak to drive directly up the hill; it must build momentum by
rocking back and forth. This makes it a challenging exploration problem where
the agent receives only negative rewards until it solves the task.

In [ ]:
# ── Explore the MountainCar environment ──
env = gym.make('MountainCar-v0')

print(f'Observation space: {env.observation_space}')
print(f'  Low:  {env.observation_space.low}')
print(f'  High: {env.observation_space.high}')
print(f'Action space: {env.action_space}')
print(f'  Number of actions: {env.action_space.n}')

# State bounds for feature construction
STATE_BOUNDS = list(zip(env.observation_space.low, env.observation_space.high))
N_ACTIONS = env.action_space.n

print(f'\nState bounds: {STATE_BOUNDS}')
print(f'Number of actions: {N_ACTIONS}')

---
## 6. Tile Coding Implementation

In [ ]:
class TileCoding:
    """Tile coding feature constructor with multiple overlapping tilings.

    Each tiling partitions the state space into a grid of tiles. Multiple
    tilings are offset from each other so that nearby states share some
    (but not all) active tiles, enabling smooth generalization.

    Args:
        n_tilings: Number of overlapping tilings.
        n_tiles_per_dim: Number of tiles along each dimension per tiling.
        state_bounds: List of (low, high) tuples, one per state dimension.

    Attributes:
        n_features: Total number of tile features across all tilings.
    """

    def __init__(
        self,
        n_tilings: int,
        n_tiles_per_dim: int,
        state_bounds: List[Tuple[float, float]],
    ):
        self.n_tilings = n_tilings
        self.n_tiles_per_dim = n_tiles_per_dim
        self.state_bounds = np.array(state_bounds, dtype=np.float64)
        self.n_dims = len(state_bounds)

        # Tile widths along each dimension
        self.tile_widths = (
            (self.state_bounds[:, 1] - self.state_bounds[:, 0]) / self.n_tiles_per_dim
        )

        # Compute offsets for each tiling.
        # Tiling k is offset by (k / n_tilings) * tile_width along each dim.
        self.offsets = np.array([
            self.tile_widths * k / self.n_tilings for k in range(self.n_tilings)
        ])

        # Total features = n_tilings * (n_tiles_per_dim ^ n_dims)
        self._tiles_per_tiling = self.n_tiles_per_dim ** self.n_dims

    @property
    def n_features(self) -> int:
        """Total number of tile features across all tilings."""
        return self.n_tilings * self._tiles_per_tiling

    def _get_tile_indices(self, state: np.ndarray, tiling_idx: int) -> int:
        """Get the linear tile index for a state within a single tiling.

        Args:
            state: State vector.
            tiling_idx: Which tiling (0 to n_tilings-1).

        Returns:
            Linear index of the activated tile within this tiling.
        """
        # Shift the state by the tiling's offset
        shifted = state - self.state_bounds[:, 0] + self.offsets[tiling_idx]
        # Determine which tile along each dimension
        tile_coords = np.clip(
            (shifted / self.tile_widths).astype(int), 0, self.n_tiles_per_dim - 1
        )
        # Convert multi-dim tile coordinates to a single linear index
        linear_idx = 0
        for d in range(self.n_dims):
            linear_idx = linear_idx * self.n_tiles_per_dim + tile_coords[d]
        return linear_idx

    def get_active_tiles(self, state: np.ndarray) -> List[int]:
        """Return the list of active tile indices (one per tiling).

        Args:
            state: State vector.

        Returns:
            List of active feature indices (length = n_tilings).
        """
        active = []
        for k in range(self.n_tilings):
            tile_idx = self._get_tile_indices(state, k)
            # Offset by tiling number so indices are globally unique
            active.append(k * self._tiles_per_tiling + tile_idx)
        return active

    def get_features(self, state: np.ndarray) -> np.ndarray:
        """Construct the binary feature vector for a given state.

        Args:
            state: State vector.

        Returns:
            Binary feature vector of shape (n_features,).
        """
        features = np.zeros(self.n_features)
        for idx in self.get_active_tiles(state):
            features[idx] = 1.0
        return features


# ── Quick sanity check ──
tc = TileCoding(n_tilings=8, n_tiles_per_dim=8, state_bounds=STATE_BOUNDS)
test_state = np.array([-0.5, 0.0])
active = tc.get_active_tiles(test_state)
feat = tc.get_features(test_state)

print(f'Number of tilings: {tc.n_tilings}')
print(f'Tiles per dim: {tc.n_tiles_per_dim}')
print(f'Total features: {tc.n_features}')
print(f'Active tiles for state {test_state}: {active}')
print(f'Number of active features: {int(feat.sum())} (should equal n_tilings = {tc.n_tilings})')

---
## 7. Tile Coding Visualization

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Show the first 3 tilings and how they overlap
tc_vis = TileCoding(n_tilings=4, n_tiles_per_dim=4, state_bounds=STATE_BOUNDS)
tiling_colors = [COLORS['blue'], COLORS['red'], COLORS['green'], COLORS['yellow']]

for ax_idx in range(3):
    ax = axes[ax_idx]
    if ax_idx < 2:
        # Show individual tilings
        k = ax_idx
        color = tiling_colors[k]
        lows = tc_vis.state_bounds[:, 0]
        highs = tc_vis.state_bounds[:, 1]
        offsets = tc_vis.offsets[k]

        # Vertical lines
        for i in range(tc_vis.n_tiles_per_dim + 1):
            x = lows[0] + offsets[0] + i * tc_vis.tile_widths[0]
            ax.axvline(x, color=color, alpha=0.7, linewidth=1.5)
        # Horizontal lines
        for j in range(tc_vis.n_tiles_per_dim + 1):
            y = lows[1] + offsets[1] + j * tc_vis.tile_widths[1]
            ax.axhline(y, color=color, alpha=0.7, linewidth=1.5)

        ax.set_xlim(lows[0] - 0.1, highs[0] + 0.1)
        ax.set_ylim(lows[1] - 0.01, highs[1] + 0.01)
        ax.set_title(f'Tiling {k + 1}', fontsize=13)
        ax.set_xlabel('Position')
        ax.set_ylabel('Velocity')
        ax.plot(-0.5, 0.0, 'k*', markersize=14, zorder=5, label='test state')
        ax.legend(loc='upper right', fontsize=10)
    else:
        # Overlay first 4 tilings
        for k in range(4):
            color = tiling_colors[k]
            lows = tc_vis.state_bounds[:, 0]
            offsets = tc_vis.offsets[k]
            for i in range(tc_vis.n_tiles_per_dim + 1):
                x = lows[0] + offsets[0] + i * tc_vis.tile_widths[0]
                ax.axvline(x, color=color, alpha=0.4, linewidth=1)
            for j in range(tc_vis.n_tiles_per_dim + 1):
                y = lows[1] + offsets[1] + j * tc_vis.tile_widths[1]
                ax.axhline(y, color=color, alpha=0.4, linewidth=1)
        ax.set_xlim(lows[0] - 0.1, tc_vis.state_bounds[0, 1] + 0.1)
        ax.set_ylim(lows[1] - 0.01, tc_vis.state_bounds[1, 1] + 0.01)
        ax.set_title('All 4 Tilings Overlaid', fontsize=13)
        ax.set_xlabel('Position')
        ax.set_ylabel('Velocity')
        ax.plot(-0.5, 0.0, 'k*', markersize=14, zorder=5)

plt.suptitle('Tile Coding: Overlapping Tilings on the MountainCar State Space', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

---
## 8. RBF Features Implementation

In [ ]:
class RBFFeatures:
    """Radial Basis Function feature constructor.

    Places Gaussian kernels on a regular grid over the state space and
    returns normalized activation values as features.

    Args:
        n_centers_per_dim: Number of RBF centers along each state dimension.
        state_bounds: List of (low, high) tuples, one per state dimension.
        sigma: Width (standard deviation) of each Gaussian kernel.

    Attributes:
        n_features: Total number of RBF features.
        centers: Array of shape (n_features, n_dims) with center locations.
    """

    def __init__(
        self,
        n_centers_per_dim: int,
        state_bounds: List[Tuple[float, float]],
        sigma: float = 0.1,
    ):
        self.n_centers_per_dim = n_centers_per_dim
        self.state_bounds = np.array(state_bounds, dtype=np.float64)
        self.sigma = sigma
        self.n_dims = len(state_bounds)

        # Build a regular grid of centers
        grids_per_dim = [
            np.linspace(lo, hi, n_centers_per_dim)
            for lo, hi in state_bounds
        ]
        mesh = np.array(list(product(*grids_per_dim)))
        self.centers = mesh  # shape (n_features, n_dims)

        # Precompute normalisation constant for the exponent
        self._two_sigma_sq = 2.0 * sigma ** 2

    @property
    def n_features(self) -> int:
        """Total number of RBF centers."""
        return self.centers.shape[0]

    def get_features(self, state: np.ndarray) -> np.ndarray:
        """Compute normalized RBF activations for a given state.

        Args:
            state: State vector of shape (n_dims,).

        Returns:
            Normalized feature vector of shape (n_features,).
            Components sum to approximately 1.
        """
        # Normalize state to [0, 1] for consistent distance computation
        ranges = self.state_bounds[:, 1] - self.state_bounds[:, 0]
        state_norm = (state - self.state_bounds[:, 0]) / ranges
        centers_norm = (self.centers - self.state_bounds[:, 0]) / ranges

        # Squared distances to each center
        diffs = centers_norm - state_norm
        sq_dists = np.sum(diffs ** 2, axis=1)

        # Gaussian activations
        activations = np.exp(-sq_dists / self._two_sigma_sq)

        # Normalize so features sum to 1
        total = activations.sum()
        if total > 1e-12:
            activations /= total

        return activations


# ── Quick sanity check ──
rbf = RBFFeatures(n_centers_per_dim=8, state_bounds=STATE_BOUNDS, sigma=0.15)
rbf_feat = rbf.get_features(test_state)

print(f'Number of RBF centers: {rbf.n_features}')
print(f'Feature vector shape: {rbf_feat.shape}')
print(f'Feature sum: {rbf_feat.sum():.6f} (should be ~1.0)')
print(f'Max activation: {rbf_feat.max():.6f}')
print(f'Non-zero features (>1e-6): {np.sum(rbf_feat > 1e-6)}')

---
## 9. RBF Feature Map Visualization

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5))

# Create a grid over state space for visualization
pos_grid = np.linspace(STATE_BOUNDS[0][0], STATE_BOUNDS[0][1], 80)
vel_grid = np.linspace(STATE_BOUNDS[1][0], STATE_BOUNDS[1][1], 80)
POS, VEL = np.meshgrid(pos_grid, vel_grid)

rbf_vis = RBFFeatures(n_centers_per_dim=6, state_bounds=STATE_BOUNDS, sigma=0.15)

# Show activation of three different centers
sample_centers = [0, rbf_vis.n_features // 2, rbf_vis.n_features - 1]
for ax_idx, ci in enumerate(sample_centers):
    ax = axes[ax_idx]
    activation_map = np.zeros_like(POS)
    for i in range(POS.shape[0]):
        for j in range(POS.shape[1]):
            s = np.array([POS[i, j], VEL[i, j]])
            # Raw (unnormalized) activation for this center
            ranges = rbf_vis.state_bounds[:, 1] - rbf_vis.state_bounds[:, 0]
            s_n = (s - rbf_vis.state_bounds[:, 0]) / ranges
            c_n = (rbf_vis.centers[ci] - rbf_vis.state_bounds[:, 0]) / ranges
            sq_d = np.sum((s_n - c_n) ** 2)
            activation_map[i, j] = np.exp(-sq_d / rbf_vis._two_sigma_sq)

    im = ax.pcolormesh(POS, VEL, activation_map, cmap='YlOrRd', shading='auto')
    ax.plot(rbf_vis.centers[ci, 0], rbf_vis.centers[ci, 1], 'k+', markersize=14, mew=3)
    ax.set_title(f'RBF Center {ci}\nc = ({rbf_vis.centers[ci, 0]:.2f}, {rbf_vis.centers[ci, 1]:.3f})',
                 fontsize=11)
    ax.set_xlabel('Position')
    ax.set_ylabel('Velocity')
    plt.colorbar(im, ax=ax, fraction=0.046)

plt.suptitle('RBF Feature Activations (Gaussian Bumps) in State Space', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

---
## 10. Feature Activation Patterns for Sample States

In [ ]:
# Compare feature activations across sample states
sample_states = [
    np.array([-1.0, 0.0]),   # far left, zero velocity
    np.array([-0.5, 0.0]),   # middle, zero velocity
    np.array([0.4, 0.05]),   # near goal, moving right
]
state_labels = ['Far left (−1.0, 0.0)', 'Middle (−0.5, 0.0)', 'Near goal (0.4, 0.05)']

fig, axes = plt.subplots(2, 3, figsize=(16, 8))

tc_check = TileCoding(n_tilings=8, n_tiles_per_dim=8, state_bounds=STATE_BOUNDS)
rbf_check = RBFFeatures(n_centers_per_dim=8, state_bounds=STATE_BOUNDS, sigma=0.15)

for col, (s, label) in enumerate(zip(sample_states, state_labels)):
    # Tile coding features (show active indices)
    tc_feat = tc_check.get_features(s)
    active_idx = np.where(tc_feat > 0)[0]
    ax = axes[0, col]
    ax.bar(range(len(active_idx)), [1.0] * len(active_idx),
           color=COLORS['blue'], alpha=0.8)
    ax.set_title(f'Tile Coding\n{label}', fontsize=11)
    ax.set_xlabel('Active tile index (within list)')
    ax.set_ylabel('Activation')
    ax.set_ylim(0, 1.5)

    # RBF features (show activation strengths)
    rbf_feat = rbf_check.get_features(s)
    ax = axes[1, col]
    # Show top 15 activations for clarity
    top_k = 15
    top_indices = np.argsort(rbf_feat)[-top_k:]
    ax.bar(range(top_k), rbf_feat[top_indices],
           color=COLORS['red'], alpha=0.8)
    ax.set_title(f'RBF (top {top_k})\n{label}', fontsize=11)
    ax.set_xlabel('Center index (sorted by activation)')
    ax.set_ylabel('Activation')

plt.suptitle('Feature Activation Patterns: Tile Coding vs RBF', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

---
## 11. Linear Value Function

In [ ]:
class LinearValueFunction:
    """Linear value function approximator: v_hat(s) = w^T x(s).

    Args:
        n_features: Dimensionality of the feature vector.

    Attributes:
        w: Weight vector of shape (n_features,).
    """

    def __init__(self, n_features: int):
        self.w = np.zeros(n_features)

    def value(self, features: np.ndarray) -> float:
        """Compute the estimated value for a given feature vector.

        Args:
            features: Feature vector of shape (n_features,).

        Returns:
            Scalar estimated value.
        """
        return float(self.w @ features)

    def update(self, features: np.ndarray, td_error: float, alpha: float) -> None:
        """Semi-gradient update: w <- w + alpha * td_error * x(s).

        Args:
            features: Feature vector x(s) of shape (n_features,).
            td_error: The TD error delta = R + gamma * v(s') - v(s).
            alpha: Learning rate.
        """
        self.w += alpha * td_error * features


print('LinearValueFunction class defined.')

---
## 12. Semi-Gradient TD(0) for Value Prediction

In [ ]:
def semi_gradient_td0(
    env: gym.Env,
    feature_fn,
    n_episodes: int,
    alpha: float,
    gamma: float,
    policy: Optional[callable] = None,
) -> Tuple[np.ndarray, List[int]]:
    """Semi-gradient TD(0) for value prediction under a fixed policy.

    If no policy is provided, uses a random policy.

    Args:
        env: Gymnasium environment.
        feature_fn: Feature constructor with get_features(state) and n_features.
        n_episodes: Number of episodes to train.
        alpha: Learning rate.
        gamma: Discount factor.
        policy: Callable that maps state -> action. Defaults to random.

    Returns:
        weights: Learned weight vector.
        episode_lengths: List of episode lengths.
    """
    vf = LinearValueFunction(feature_fn.n_features)
    episode_lengths = []

    if policy is None:
        policy = lambda s: env.action_space.sample()

    for ep in range(n_episodes):
        state, _ = env.reset(seed=SEED + ep)
        x = feature_fn.get_features(state)
        done = False
        steps = 0

        while not done:
            action = policy(state)
            next_state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            steps += 1

            x_next = feature_fn.get_features(next_state)
            v_next = 0.0 if terminated else vf.value(x_next)
            td_error = reward + gamma * v_next - vf.value(x)
            vf.update(x, td_error, alpha)

            state = next_state
            x = x_next

        episode_lengths.append(steps)

    return vf.w, episode_lengths


print('semi_gradient_td0 defined.')

---
## 13. Semi-Gradient SARSA for Control

In [ ]:
def epsilon_greedy_action(q_values: np.ndarray, epsilon: float) -> int:
    """Select an action using epsilon-greedy strategy.

    Args:
        q_values: Array of Q-values for each action.
        epsilon: Exploration probability.

    Returns:
        Selected action index.
    """
    if np.random.random() < epsilon:
        return np.random.randint(len(q_values))
    return int(np.argmax(q_values))


def semi_gradient_sarsa(
    env: gym.Env,
    feature_fn,
    n_actions: int,
    n_episodes: int,
    alpha: float,
    gamma: float,
    epsilon: float,
) -> Tuple[np.ndarray, List[int], List[float]]:
    """Semi-gradient SARSA for control with linear function approximation.

    Uses one weight vector per action: Q(s, a) = w_a^T x(s).

    Args:
        env: Gymnasium environment.
        feature_fn: Feature constructor with get_features(state) and n_features.
        n_actions: Number of discrete actions.
        n_episodes: Number of episodes to train.
        alpha: Learning rate.
        gamma: Discount factor.
        epsilon: Exploration rate for epsilon-greedy.

    Returns:
        weights: Weight matrix of shape (n_actions, n_features).
        episode_lengths: List of episode lengths.
        episode_rewards: List of total rewards per episode.
    """
    # One weight vector per action
    weights = np.zeros((n_actions, feature_fn.n_features))
    episode_lengths = []
    episode_rewards = []

    for ep in range(n_episodes):
        state, _ = env.reset(seed=SEED + ep)
        x = feature_fn.get_features(state)

        # Q-values for current state
        q_vals = weights @ x
        action = epsilon_greedy_action(q_vals, epsilon)

        done = False
        total_reward = 0.0
        steps = 0

        while not done:
            next_state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            total_reward += reward
            steps += 1

            x_next = feature_fn.get_features(next_state)

            if terminated:
                # Terminal state: TD target is just the reward
                td_error = reward - (weights[action] @ x)
                weights[action] += alpha * td_error * x
            else:
                # Choose next action (SARSA is on-policy)
                q_next = weights @ x_next
                next_action = epsilon_greedy_action(q_next, epsilon)

                td_error = reward + gamma * (weights[next_action] @ x_next) - (weights[action] @ x)
                weights[action] += alpha * td_error * x

                action = next_action

            state = next_state
            x = x_next

        episode_lengths.append(steps)
        episode_rewards.append(total_reward)

        if (ep + 1) % 200 == 0:
            avg_len = np.mean(episode_lengths[-50:])
            print(f'  Episode {ep + 1:5d} | Avg length (last 50): {avg_len:.1f}')

    return weights, episode_lengths, episode_rewards


print('epsilon_greedy_action and semi_gradient_sarsa defined.')

---
## 14. Train SARSA with Tile Coding on MountainCar

In [ ]:
print('Training Semi-Gradient SARSA with Tile Coding ...')
print('=' * 55)

tc_sarsa = TileCoding(n_tilings=8, n_tiles_per_dim=8, state_bounds=STATE_BOUNDS)
env_train = gym.make('MountainCar-v0')

np.random.seed(SEED)
tc_weights, tc_lengths, tc_rewards = semi_gradient_sarsa(
    env=env_train,
    feature_fn=tc_sarsa,
    n_actions=N_ACTIONS,
    n_episodes=N_EPISODES,
    alpha=ALPHA / tc_sarsa.n_tilings,  # scale alpha by number of active features
    gamma=GAMMA,
    epsilon=EPSILON,
)

print(f'\nFinal avg episode length (last 100): {np.mean(tc_lengths[-100:]):.1f}')

---
## 15. Train SARSA with RBF Features on MountainCar

In [ ]:
print('Training Semi-Gradient SARSA with RBF Features ...')
print('=' * 55)

rbf_sarsa = RBFFeatures(n_centers_per_dim=8, state_bounds=STATE_BOUNDS, sigma=0.10)

np.random.seed(SEED)
rbf_weights, rbf_lengths, rbf_rewards = semi_gradient_sarsa(
    env=env_train,
    feature_fn=rbf_sarsa,
    n_actions=N_ACTIONS,
    n_episodes=N_EPISODES,
    alpha=ALPHA,
    gamma=GAMMA,
    epsilon=EPSILON,
)

print(f'\nFinal avg episode length (last 100): {np.mean(rbf_lengths[-100:]):.1f}')

---
## 16. Learning Curves: Tile Coding vs RBF

In [ ]:
def smooth(data, window=30):
    """Simple moving average for smoothing."""
    return np.convolve(data, np.ones(window) / window, mode='valid')


fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Episode length over time
ax = axes[0]
ax.plot(smooth(tc_lengths), color=COLORS['blue'], label='Tile Coding (8 tilings)', alpha=0.9)
ax.plot(smooth(rbf_lengths), color=COLORS['red'], label='RBF Features', alpha=0.9)
ax.set_xlabel('Episode')
ax.set_ylabel('Episode Length')
ax.set_title('Episode Length vs Episode')
ax.legend(fontsize=11)
ax.set_ylim(50, 210)

# Cumulative reward over time
ax = axes[1]
ax.plot(smooth(tc_rewards), color=COLORS['blue'], label='Tile Coding', alpha=0.9)
ax.plot(smooth(rbf_rewards), color=COLORS['red'], label='RBF Features', alpha=0.9)
ax.set_xlabel('Episode')
ax.set_ylabel('Total Reward')
ax.set_title('Reward vs Episode')
ax.legend(fontsize=11)

plt.suptitle('Learning Curves: Semi-Gradient SARSA on MountainCar', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

---
## 17. Learned Value Surface (Tile Coding)

In [ ]:
def compute_value_surface(weights, feature_fn, n_actions, state_bounds, resolution=50):
    """Compute V(s) = max_a Q(s,a) over a grid of states.

    Args:
        weights: Weight matrix of shape (n_actions, n_features).
        feature_fn: Feature constructor.
        n_actions: Number of actions.
        state_bounds: List of (low, high) tuples.
        resolution: Grid resolution per dimension.

    Returns:
        POS, VEL: Meshgrid arrays.
        V: Value surface of shape (resolution, resolution).
    """
    pos = np.linspace(state_bounds[0][0], state_bounds[0][1], resolution)
    vel = np.linspace(state_bounds[1][0], state_bounds[1][1], resolution)
    POS, VEL = np.meshgrid(pos, vel)
    V = np.zeros_like(POS)

    for i in range(resolution):
        for j in range(resolution):
            s = np.array([POS[i, j], VEL[i, j]])
            x = feature_fn.get_features(s)
            q_vals = weights @ x
            V[i, j] = np.max(q_vals)

    return POS, VEL, V


POS_tc, VEL_tc, V_tc = compute_value_surface(
    tc_weights, tc_sarsa, N_ACTIONS, STATE_BOUNDS, resolution=60
)

fig = plt.figure(figsize=(14, 5))

# 3D surface
ax1 = fig.add_subplot(121, projection='3d')
ax1.plot_surface(POS_tc, VEL_tc, V_tc, cmap='viridis', alpha=0.9, edgecolor='none')
ax1.set_xlabel('Position')
ax1.set_ylabel('Velocity')
ax1.set_zlabel('V(s)')
ax1.set_title('Value Surface (Tile Coding)', fontsize=13)
ax1.view_init(elev=30, azim=225)

# 2D heatmap
ax2 = fig.add_subplot(122)
im = ax2.pcolormesh(POS_tc, VEL_tc, V_tc, cmap='viridis', shading='auto')
ax2.set_xlabel('Position')
ax2.set_ylabel('Velocity')
ax2.set_title('Value Heatmap (Tile Coding)', fontsize=13)
ax2.axvline(0.5, color='red', linestyle='--', linewidth=2, label='Goal')
ax2.legend(fontsize=11)
plt.colorbar(im, ax=ax2, label='V(s)')

plt.tight_layout()
plt.show()

---
## 18. Learned Value Surface (RBF)

In [ ]:
POS_rbf, VEL_rbf, V_rbf = compute_value_surface(
    rbf_weights, rbf_sarsa, N_ACTIONS, STATE_BOUNDS, resolution=60
)

fig = plt.figure(figsize=(14, 5))

# 3D surface
ax1 = fig.add_subplot(121, projection='3d')
ax1.plot_surface(POS_rbf, VEL_rbf, V_rbf, cmap='plasma', alpha=0.9, edgecolor='none')
ax1.set_xlabel('Position')
ax1.set_ylabel('Velocity')
ax1.set_zlabel('V(s)')
ax1.set_title('Value Surface (RBF Features)', fontsize=13)
ax1.view_init(elev=30, azim=225)

# 2D heatmap
ax2 = fig.add_subplot(122)
im = ax2.pcolormesh(POS_rbf, VEL_rbf, V_rbf, cmap='plasma', shading='auto')
ax2.set_xlabel('Position')
ax2.set_ylabel('Velocity')
ax2.set_title('Value Heatmap (RBF Features)', fontsize=13)
ax2.axvline(0.5, color='red', linestyle='--', linewidth=2, label='Goal')
ax2.legend(fontsize=11)
plt.colorbar(im, ax=ax2, label='V(s)')

plt.tight_layout()
plt.show()

---
## 19. Polynomial Features Baseline

In [ ]:
class PolynomialFeatures:
    """Simple polynomial feature constructor for baseline comparison.

    Generates all monomials up to a given degree, with state normalized
    to [0, 1].

    Args:
        degree: Maximum polynomial degree.
        state_bounds: List of (low, high) tuples, one per state dimension.

    Attributes:
        n_features: Total number of polynomial features.
    """

    def __init__(self, degree: int, state_bounds: List[Tuple[float, float]]):
        self.degree = degree
        self.state_bounds = np.array(state_bounds, dtype=np.float64)
        self.n_dims = len(state_bounds)

        # Enumerate all monomial exponents with total degree <= degree
        self._exponents = []
        for total_deg in range(degree + 1):
            for p in range(total_deg + 1):
                q = total_deg - p
                if q >= 0:
                    self._exponents.append((p, q))
        self._exponents = np.array(self._exponents)

    @property
    def n_features(self) -> int:
        return len(self._exponents)

    def get_features(self, state: np.ndarray) -> np.ndarray:
        """Compute polynomial features for a given state.

        Args:
            state: State vector.

        Returns:
            Feature vector of shape (n_features,).
        """
        # Normalize to [0, 1]
        ranges = self.state_bounds[:, 1] - self.state_bounds[:, 0]
        s_norm = (state - self.state_bounds[:, 0]) / ranges

        features = np.ones(self.n_features)
        for i, (p, q) in enumerate(self._exponents):
            features[i] = (s_norm[0] ** p) * (s_norm[1] ** q)
        return features


poly = PolynomialFeatures(degree=5, state_bounds=STATE_BOUNDS)
print(f'Polynomial features (degree 5): {poly.n_features} features')
print(f'Sample: {poly.get_features(np.array([-0.5, 0.0]))[:6].round(4)}')

---
## 20. Train SARSA with Polynomial Features

In [ ]:
print('Training Semi-Gradient SARSA with Polynomial Features ...')
print('=' * 55)

poly_sarsa = PolynomialFeatures(degree=5, state_bounds=STATE_BOUNDS)

np.random.seed(SEED)
poly_weights, poly_lengths, poly_rewards = semi_gradient_sarsa(
    env=env_train,
    feature_fn=poly_sarsa,
    n_actions=N_ACTIONS,
    n_episodes=N_EPISODES,
    alpha=0.001,  # smaller step size for stability with polynomial features
    gamma=GAMMA,
    epsilon=EPSILON,
)

print(f'\nFinal avg episode length (last 100): {np.mean(poly_lengths[-100:]):.1f}')

---
## 21. Convergence Comparison: Tile Coding vs RBF vs Polynomial

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Episode lengths
ax = axes[0]
ax.plot(smooth(tc_lengths, 50), color=COLORS['blue'], label='Tile Coding (8x8, 8 tilings)', alpha=0.9)
ax.plot(smooth(rbf_lengths, 50), color=COLORS['red'], label='RBF (8x8 centers)', alpha=0.9)
ax.plot(smooth(poly_lengths, 50), color=COLORS['green'], label='Polynomial (degree 5)', alpha=0.9)
ax.axhline(200, color='gray', linestyle=':', alpha=0.5, label='Max steps (200)')
ax.set_xlabel('Episode')
ax.set_ylabel('Episode Length (smoothed)')
ax.set_title('Episode Length Comparison')
ax.legend(fontsize=10)
ax.set_ylim(50, 210)

# Cumulative reward
ax = axes[1]
ax.plot(smooth(tc_rewards, 50), color=COLORS['blue'], label='Tile Coding', alpha=0.9)
ax.plot(smooth(rbf_rewards, 50), color=COLORS['red'], label='RBF', alpha=0.9)
ax.plot(smooth(poly_rewards, 50), color=COLORS['green'], label='Polynomial', alpha=0.9)
ax.set_xlabel('Episode')
ax.set_ylabel('Total Reward (smoothed)')
ax.set_title('Reward Comparison')
ax.legend(fontsize=10)

plt.suptitle('Convergence Comparison of Feature Representations', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

---
## 22. Single-Tiling Baseline (Tile Coding Ablation)

In [ ]:
print('Training with a SINGLE tiling (no overlap) ...')
print('=' * 50)

tc_single = TileCoding(n_tilings=1, n_tiles_per_dim=8, state_bounds=STATE_BOUNDS)

np.random.seed(SEED)
single_weights, single_lengths, single_rewards = semi_gradient_sarsa(
    env=env_train,
    feature_fn=tc_single,
    n_actions=N_ACTIONS,
    n_episodes=N_EPISODES,
    alpha=ALPHA,
    gamma=GAMMA,
    epsilon=EPSILON,
)

print(f'\nSingle tiling -- final avg length (last 100): {np.mean(single_lengths[-100:]):.1f}')
print(f'8 tilings     -- final avg length (last 100): {np.mean(tc_lengths[-100:]):.1f}')

---
## 23. Single Tiling vs Multi-Tiling Comparison

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

ax.plot(smooth(single_lengths, 50), color=COLORS['yellow'], label='Single tiling (1x8x8)',
        alpha=0.9, linewidth=2.5)
ax.plot(smooth(tc_lengths, 50), color=COLORS['blue'], label='8 tilings (8x8x8)',
        alpha=0.9, linewidth=2.5)
ax.axhline(200, color='gray', linestyle=':', alpha=0.5, label='Max steps (200)')
ax.set_xlabel('Episode')
ax.set_ylabel('Episode Length (smoothed)')
ax.set_title('Effect of Multiple Tilings: Overlap Enables Generalization', fontsize=13)
ax.legend(fontsize=11)
ax.set_ylim(50, 210)

plt.tight_layout()
plt.show()

---
## 24. Semi-Gradient TD(0) Value Prediction Demo

In [ ]:
# Train TD(0) value prediction under the learned tile-coding policy
def greedy_tc_policy(state):
    """Greedy policy derived from the tile-coding SARSA weights."""
    x = tc_sarsa.get_features(state)
    q_vals = tc_weights @ x
    return int(np.argmax(q_vals))


print('Running Semi-Gradient TD(0) for value prediction ...')
tc_td0 = TileCoding(n_tilings=8, n_tiles_per_dim=8, state_bounds=STATE_BOUNDS)

np.random.seed(SEED)
td0_weights, td0_lengths = semi_gradient_td0(
    env=env_train,
    feature_fn=tc_td0,
    n_episodes=500,
    alpha=ALPHA / 8,
    gamma=GAMMA,
    policy=greedy_tc_policy,
)

# Plot the learned V(s)
pos_g = np.linspace(STATE_BOUNDS[0][0], STATE_BOUNDS[0][1], 50)
vel_g = np.linspace(STATE_BOUNDS[1][0], STATE_BOUNDS[1][1], 50)
PG, VG = np.meshgrid(pos_g, vel_g)
V_td0 = np.zeros_like(PG)
for i in range(50):
    for j in range(50):
        s = np.array([PG[i, j], VG[i, j]])
        V_td0[i, j] = td0_weights @ tc_td0.get_features(s)

fig = plt.figure(figsize=(14, 5))

ax1 = fig.add_subplot(121, projection='3d')
ax1.plot_surface(PG, VG, V_td0, cmap='coolwarm', alpha=0.9, edgecolor='none')
ax1.set_xlabel('Position')
ax1.set_ylabel('Velocity')
ax1.set_zlabel('V(s)')
ax1.set_title('TD(0) Value Surface Under Learned Policy', fontsize=12)
ax1.view_init(elev=30, azim=225)

ax2 = fig.add_subplot(122)
im = ax2.pcolormesh(PG, VG, V_td0, cmap='coolwarm', shading='auto')
ax2.set_xlabel('Position')
ax2.set_ylabel('Velocity')
ax2.set_title('TD(0) Value Heatmap', fontsize=12)
ax2.axvline(0.5, color='black', linestyle='--', linewidth=2, label='Goal')
ax2.legend(fontsize=11)
plt.colorbar(im, ax=ax2, label='V(s)')

plt.tight_layout()
plt.show()

---
## 25. Verification

In [ ]:
print('=' * 60)
print('VERIFICATION CHECKS')
print('=' * 60)

# ── Check 1: Tile coding feature count ──
tc_test = TileCoding(n_tilings=8, n_tiles_per_dim=8, state_bounds=STATE_BOUNDS)
expected_features = 8 * (8 ** 2)  # n_tilings * n_tiles_per_dim^n_dims
actual_features = tc_test.n_features
check1 = actual_features == expected_features
print(f'\n1. Tile coding produces correct number of features:')
print(f'   Expected: {expected_features}, Got: {actual_features}')
print(f'   {"[PASS]" if check1 else "[FAIL]"}')

# Also verify number of active features
test_feat = tc_test.get_features(np.array([-0.3, 0.02]))
n_active = int(test_feat.sum())
check1b = n_active == tc_test.n_tilings
print(f'   Active features = {n_active} (should be {tc_test.n_tilings})')
print(f'   {"[PASS]" if check1b else "[FAIL]"}')

# ── Check 2: RBF features sum to ~1 ──
rbf_test = RBFFeatures(n_centers_per_dim=8, state_bounds=STATE_BOUNDS, sigma=0.15)
test_states = [
    np.array([-1.0, 0.0]),
    np.array([-0.5, 0.03]),
    np.array([0.0, -0.05]),
    np.array([0.4, 0.06]),
]
rbf_sums = [rbf_test.get_features(s).sum() for s in test_states]
check2 = all(abs(s - 1.0) < 0.01 for s in rbf_sums)
print(f'\n2. RBF features sum to approximately 1 (normalized):')
for s, rs in zip(test_states, rbf_sums):
    print(f'   State {s} -> sum = {rs:.6f}')
print(f'   {"[PASS]" if check2 else "[FAIL]"}')

# ── Check 3: Agent solves MountainCar ──
# Check if episode length < 200 consistently in the last 100 episodes
tc_last100_avg = np.mean(tc_lengths[-100:])
check3 = tc_last100_avg < 200
print(f'\n3. Agent solves MountainCar (episode length < 200 consistently):')
print(f'   Tile Coding: avg last 100 episodes = {tc_last100_avg:.1f}')
print(f'   {"[PASS]" if check3 else "[FAIL]"}')

# ── Check 4: Value function shows high values near goal ──
# States near goal position (>= 0.5) should have higher (less negative) values
goal_states = [np.array([0.5, 0.03]), np.array([0.45, 0.05])]
far_states = [np.array([-1.0, 0.0]), np.array([-0.8, -0.03])]

goal_vals = []
far_vals = []
for s in goal_states:
    x = tc_sarsa.get_features(s)
    goal_vals.append(np.max(tc_weights @ x))
for s in far_states:
    x = tc_sarsa.get_features(s)
    far_vals.append(np.max(tc_weights @ x))

avg_goal_v = np.mean(goal_vals)
avg_far_v = np.mean(far_vals)
check4 = avg_goal_v > avg_far_v
print(f'\n4. Value function shows high values near goal:')
print(f'   Avg V near goal: {avg_goal_v:.2f}')
print(f'   Avg V far from goal: {avg_far_v:.2f}')
print(f'   Near-goal > Far-from-goal: {check4}')
print(f'   {"[PASS]" if check4 else "[FAIL]"}')

# ── Check 5: Tile coding outperforms single wide tiling ──
tc_multi_avg = np.mean(tc_lengths[-100:])
tc_single_avg = np.mean(single_lengths[-100:])
check5 = tc_multi_avg < tc_single_avg
print(f'\n5. Tile coding (8 tilings) outperforms single tiling:')
print(f'   8 tilings avg length: {tc_multi_avg:.1f}')
print(f'   1 tiling avg length:  {tc_single_avg:.1f}')
print(f'   {"[PASS]" if check5 else "[FAIL]"}')

print('\n' + '=' * 60)
all_pass = all([check1, check1b, check2, check3, check4, check5])
print(f'Overall: {"ALL CHECKS PASSED" if all_pass else "SOME CHECKS FAILED"}')
print('=' * 60)

---
## 26. Summary & Key Takeaways

### What We Built

| Component | Description |
|-----------|-------------|
| `TileCoding` | Binary features via overlapping grid tilings |
| `RBFFeatures` | Soft Gaussian activations from a grid of centers |
| `PolynomialFeatures` | Polynomial basis (baseline) |
| `LinearValueFunction` | $\hat{v}(s) = \mathbf{w}^\top \mathbf{x}(s)$ |
| `semi_gradient_td0` | Value prediction under a fixed policy |
| `semi_gradient_sarsa` | On-policy control with function approximation |

### Key Takeaways

1. **Function approximation** enables RL in continuous/large state spaces where
   tabular methods are infeasible.

2. **Tile coding** provides efficient, sparse binary features. Multiple
   overlapping tilings enable generalization across nearby states while
   maintaining computational simplicity.

3. **RBF features** offer smooth, continuous generalization through Gaussian
   kernels. The width parameter $\sigma$ controls the trade-off between
   generalization and discrimination.

4. **Semi-gradient methods** are necessary because bootstrapping targets depend
   on the weights. Despite ignoring part of the gradient, they converge
   reliably for linear function approximation.

5. **Feature engineering matters**: the choice of features (tile coding, RBF,
   polynomial) significantly affects learning speed and final performance.

6. **Multiple tilings vs single tiling**: overlapping tilings provide much
   better generalization than a single grid, leading to faster learning.

### Next Steps

- **Eligibility traces**: Combine function approximation with $n$-step or
  $\lambda$-return methods (Notebook 4)
- **Nonlinear approximation**: Replace linear $\mathbf{w}^\top \mathbf{x}$
  with neural networks (Deep Q-Networks)
- **Policy gradient methods**: Parameterize the policy directly instead of the
  value function

In [ ]:
env_train.close()
print('Notebook complete. All environments closed.')